In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import networkx as nx

from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
    basis_configs_from_build_result,
    sector_mask_from_build_result,
    extract_cage_region_support,
    distinct_reduced_iz_pattern_supports,
    build_type1_cage_lindblad_construction,
)
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.models.couplings import DirectedPlaquetteCoupling
from qlinks.visualizer import (
    LinkVisualStyle,
    BasisGridVisualizer,
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
)

# Model definition

In [ ]:
def coup_kin(plaquette_id: int) -> DirectedPlaquetteCoupling:
    if plaquette_id in [76, 77, 78]:
        phase = 0.5 * np.pi
    # elif plaquette_id in [79]:
    #     phase = -0.5 * np.pi
    else:
        phase = 0
    return DirectedPlaquetteCoupling(
        forward=-np.exp(1j * phase),
    )

In [ ]:
model = HoneycombQDMModel(
    lx=2,
    ly=2,
    boundary_condition="periodic",
    winding_x=-2,
    winding_y=-2,
    # charge_magnitude=1,
    coup_kin=-1.0,
    coup_pot=1.0,
)
print(model.allowed_sector_labels())
print(model.nonempty_sector_labels())
# print(model.lattice.plaquette_id_from_anchor((3, 2), kind="rhombus_bc"))

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("H shape =", hamiltonian_matrix.shape)
print("H nnz =", hamiltonian_matrix.nnz)
print("K nnz =", kinetic_matrix.nnz)
print("V nnz =", potential_matrix.nnz)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

# Search for caged states

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="qlm",
        # type1_kappas=(0,),
        # type2_kappas=(-2, 2),
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = 1
record = search_result[signature, record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected caged state

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
    ),
)

# print(report)
print(report.to_text(verbose=True))

## Diagnostic

In [ ]:
region = extract_cage_region_support(report)

support_R_links = region.variable_index_set
patterns = distinct_reduced_iz_pattern_supports(report)

print("R =", region.variable_indices)
print("number of variables in R =", region.region_size)
print("number of distinct reduced IZ patterns =", len(patterns))

In [ ]:
problem = build_type1_cage_lindblad_construction(
    model=model,
    build_result=build_result,
    cage_state=record.full_state,
    classification_report=report,
    z_value=signature[1],  # infer automatically
)

print("region_size", problem.region.region_size)
print("inside_plaquette_ids:", problem.inside_plaquette_ids)
print("outside_plaquette_ids:", problem.outside_plaquette_ids)
print("crossing_plaquette_ids:", problem.crossing_plaquette_ids)
print("monitor_plaquette_ids:", problem.monitor_plaquette_ids)
print("jump_plaquette_ids:", problem.jump_plaquette_ids)
print("monitor_residual", problem.monitor_residual)
print("max_jump_residual", problem.max_jump_residual)
print("liouvillian_residual", problem.liouvillian_residual)

In [ ]:
from qlinks.open_system import evolve_matrix_rk4, fidelity_pure, random_density_matrix


times = np.linspace(0.0, 20.0, 40)

rho0 = random_density_matrix(
    dim=build_result.hamiltonian.shape[0],
    kind="mixed",
    rng=0,
)

rhos = evolve_matrix_rk4(
    rho0,
    build_result.hamiltonian,
    list(problem.jumps),
    times,
    renormalize_trace=True,
    enforce_hermiticity=True,
)

fidelities = np.array([
    fidelity_pure(problem.cage_state, rho)
    for rho in rhos
])

fidelities

In [ ]:
v = record.full_state

exp_kin = v.T @ kinetic_matrix @ v
exp_pot = v.T @ potential_matrix @ v
exp_H = v.T @ hamiltonian_matrix @ v
print(exp_kin, exp_pot, exp_H)
print(np.allclose(kinetic_matrix @ v, 0, atol=1e-10), np.linalg.norm(kinetic_matrix @ v))

# Visualize the caged state on Hamiltonian graph

In [ ]:
graph_visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    kinetic_matrix,
    include_self_loops=False,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=18,
    ),
)

graph_visualizer.plot(
    backend="igraph-cairo",
    color_by="state_amplitude_real",
    state_vector=record.full_state,
    layout="kk",
    title=f"Fock-space graph colored by cage-state signed amplitude",
)

In [ ]:
small_viz = HamiltonianGraphVisualizer.cage_subgraph_from_sparse_matrix(
    build_result.kinetic,
    record.full_state,
    classification_report=report,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=24,
    ),
)

small_viz.plot(
    backend="igraph-mpl",
    color_by="state_amplitude_real",
    layout="kk",
    title=f"The relevant subgraph colored by cage-state signed amplitude",
)

### Save the graph (optional)

In [ ]:
graph_visualizer.save_graph(
    f"../data/fock_graph.gexf",
    layout_backend="igraph",
    layout="kk",
    color_by="state_amplitude_real",
    state_vector=record.full_state,
)

# Visualize the basis states for
* caged state
* interference zeros
* all basis states (optional)

In [ ]:
grid_visualizer = BasisGridVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    periodic_image_mode="positive_patch",
    collapse_duplicate_visual_links=False,
    site_label_style="sublattice_cell",
    # coordinate_transform=np.array(
    #     [
    #         [1.0, 0.0],
    #         [0.0, 0.72],
    #     ],
    #     dtype=float,
    # ),
    style=LinkVisualStyle(
        node_size=50.0,
        site_label_fontsize=4,
        link_label_fontsize=8,
        plaquette_symbol_fontsize=8,
        vulnerable_link_arrow_length_fraction=1.1,
        # plaquette_symbol_offset=(-0.01, 0.01),
    ),
)

In [ ]:
grid_visualizer.plot_cage_support(
    search_result,
    ncols=4,
    basis_configs=basis_configs,
    signature=signature,
    record_index=record_index,
    show_config_label=False,
    # figsize=(16, 8),
    # single_plot_kwargs={
    #     "with_site_labels": True,
    #     "with_site_values": False,
    #     "with_link_values": False,
    #     "with_link_ids": True,
    # },
)

In [ ]:
grid_visualizer.plot_interference_zeros(
    report,
    ncols=4,
    basis_configs=basis_configs,
    mechanism="all",
    show_config_label=False,
)

In [ ]:
grid_visualizer.plot(
    basis_configs,
    ncols=4,
    show_config_label=True,
    # suptitle="All basis configurations",
    single_plot_kwargs={
        "with_site_labels": False,
        "with_site_values": False,
        "with_link_values": False,
        "with_link_ids": False,
    },
)

In [ ]:
def _physical_qdm_resonance_count(
    visualizer: BasisConfigurationVisualizer,
    config: np.ndarray,
) -> int:
    count = 0

    for plaquette in visualizer.lattice.plaquettes:
        link_ids = tuple(int(link_id) for link_id in plaquette.links)

        if len(link_ids) < 4 or len(link_ids) % 2 != 0:
            continue

        values = [
            visualizer.link_value(config, link_id)
            for link_id in link_ids
        ]

        if visualizer._qdm_resonance_symbol(values) is not None:
            count += 1

    return count


from qlinks.visualizer import BasisConfigurationVisualizer

visualizer = BasisConfigurationVisualizer(
    lattice=model.lattice,
    layout=model.layout,
    periodic_image_mode="positive_patch",
    collapse_duplicate_visual_links=True,
)

config = basis_configs[1]
visualizer.plot(
    config,
    with_site_labels=False,
    with_site_values=False,
    with_link_values=True,
    with_link_ids=True,
)

_physical_qdm_resonance_count(visualizer, config)

## Classify all cages

In [ ]:
import pandas as pd

def classify_cage_search_result(
    search_result,
    *,
    kinetic_matrix,
    basis_configs,
    sector_mask=None,
    config=None,
) -> tuple[pd.DataFrame, list]:
    """Classify all CageRecords in a CageSearchResult.

    Returns
    -------
    df:
        One row per cage record.
    reports:
        The full CageClassificationReport objects, in the same order as df.
    """
    if config is None:
        config = CageClassificationConfig()

    rows = []
    reports = []

    for signature in search_result.signatures:
        records = search_result[signature]

        for record_index, record in enumerate(records):
            report = classify_cage_state(
                record.cage_state,
                kinetic_matrix=kinetic_matrix,
                basis_configs=basis_configs,
                hilbert_size=search_result.hilbert_size,
                sector_mask=sector_mask,
                config=config,
            )

            reports.append(report)

            rows.append(
                {
                    "signature": signature,
                    "kappa": int(signature[0]),
                    "Z": int(signature[1]),
                    "record_index": int(record_index),
                    "global_record_index": len(reports) - 1,
                    "label": report.label,
                    "energy": complex(record.cage_state.energy),
                    "support_size": int(report.support_size),
                    "support_fraction": float(report.support_fraction),
                    "n_nontrivial_zeros": int(report.n_nontrivial_zeros),
                    "n_distinct_local_patterns": int(
                        report.n_distinct_local_patterns
                    ),
                    "n_q_empty_source_probes": int(
                        report.n_q_empty_source_probes
                    ),
                    "n_closed_by_known_zero_network_source_probes": int(
                        report.n_closed_by_known_zero_network_source_probes
                    ),
                    "n_projector_like_source_probes": int(
                        report.n_projector_like_source_probes
                    ),
                    "n_invalid_source_probes": int(
                        report.n_invalid_source_probes
                    ),
                    "n_regional_source_probes": int(
                        report.n_regional_source_probes
                    ),
                    "n_unexpected_target_probe_failures": int(
                        report.n_unexpected_target_probe_failures
                    ),
                    "n_nonzero_complement_action_probe_failures": int(
                        report.n_nonzero_complement_action_probe_failures
                    ),
                    "n_source_projector_like_probes": int(
                        report.n_source_projector_like_probes
                    ),
                    "n_indirect_projector_like_probes": int(
                        report.n_indirect_projector_like_probes
                    ),
                    "mean_q_sector_weight": float(report.mean_q_sector_weight),
                    "max_q_sector_weight": float(report.max_q_sector_weight),
                    "mean_complement_action_norm": float(
                        report.mean_complement_action_norm
                    ),
                    "max_complement_action_norm": float(
                        report.max_complement_action_norm
                    ),
                    "boundary_residual": float(
                        record.cage_state.boundary_residual
                    ),
                    "eigen_residual": float(record.cage_state.eigen_residual),
                    "full_residual": float(record.cage_state.full_residual),
                }
            )

    return pd.DataFrame(rows), reports


df, classification_reports = classify_cage_search_result(
    search_result,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=None,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
        collective_cancellation_mode="all_problematic_nullspace",
    ),
)

df